In [ ]:
#%pip install pypdf
#%pip install gradio
#%pip install -U jupyter ipywidgets
%pip install -U gradio

In [2]:
from dotenv import load_dotenv
from pypdf import PdfReader
from openai import OpenAI
import gradio as gr
import os


In [3]:
load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini_client = OpenAI(
    base_url=gemini_base_url,
    api_key=gemini_api_key
)


In [4]:
def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

In [5]:
linkedin = extract_text_from_pdf("me/Amit Srivastav.pdf")
print(linkedin)



162 connections
Staff Software Engineer-I at BlueYonder India | JAVA| PGDAC CDAC  |  Azure cloud | 
Generative AI foundations | Python
Blue Yonder · Centre for Development of Advanced Computing(C-DAC)
Hyderabad, Telangana, India· Contact info
Amit Srivastav
Open to Add section Enhance profile
 ou’re open to work — you 
 his.
Share that you’re hiring and attract qualified 
candidates.
Get started
Showcase y
profile so y
Add service
Analytics
Private to you
8 profile views
Discover who’s viewed your profile.
0 post impressions
Start a post to increase engagement.
Past 7 days
11 search appearances
See how often you appear in search results.
Show all
About
12+ years of Professional Experience in Developing and Maintaining the Enterprise Applications using 
Java technologies | Oracle Certified Java Professional SE Programmer | Azure Fundamentals certified | 
Azure Developing Solutions | Spark | Databricks | Grafana | Prometheus | Codacy | Docker | Code Review 
and Quality | Team Sprit |

In [6]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

print(summary)

My name is Amit Srivastav. I'm software engineer . I'm originally from India. l foods, particularly Indian food, but strangely I'm repelled by almost all forms of Paneer Dishes. I always try to learn new technologies.


In [7]:
name = "Amit Srivastav"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [9]:
system_prompt

"You are acting as Amit Srivastav. You are answering questions on Amit Srivastav's website, particularly questions related to Amit Srivastav's career, background, skills and experience. Your responsibility is to represent Amit Srivastav for interactions on the website as faithfully as possible. You are given a summary of Amit Srivastav's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Amit Srivastav. I'm software engineer . I'm originally from India. l foods, particularly Indian food, but strangely I'm repelled by almost all forms of Paneer Dishes. I always try to learn new technologies.\n\n## LinkedIn Profile:\n\ueddb\n\ueddc\n162 connections\nStaff Software Engineer-I at BlueYonder India | JAVA| PGDAC CDAC  |  Azure cloud | \nGenerative AI foundations | Python\nBlue Yonder · Centre for

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini_client.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages)
    return response.choices[0].message.content

In [11]:
#gr.ChatInterface(chat, type="messages").launch()
gr.ChatInterface(fn=chat).launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [12]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [13]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [15]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [23]:
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini_client.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [ ]:
def chat1(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini_client.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages)
    evaluation = evaluate(response.choices[0].message.content, message, history)
    if evaluation.is_acceptable:
        return response.choices[0].message.content
    else:
        print(evaluation.feedback)
        return f"The response is not acceptable. Here's the feedback: {evaluation.feedback}"



In [33]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini_client.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat2(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini_client.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages)
    evaluation = evaluate(response.choices[0].message.content, message, history)
    if evaluation.is_acceptable:
        return response.choices[0].message.content
    else:
        print(evaluation.feedback)
        return rerun(response.choices[0].message.content, message, history, evaluation.feedback)


In [35]:
gr.ChatInterface(fn=chat1).launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


The agent's response is professional and engaging, appropriately welcoming the user to the website and offering clear options for further conversation. It aligns well with the instructed persona of Amit Srivastav speaking to a potential client or employer.
